# <b>The RL Agent

Installing libraries

## Importing Libraries

In [1]:
import torch
from torch.utils.data import Dataset
from tqdm import tqdm

from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader
from datasets import load_dataset
from torch.optim import AdamW
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import LoraConfig, TaskType, get_peft_model

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

In [2]:
device = 'cuda'

## Import the Dataset

In [3]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
dataset_path = "/content/drive/MyDrive/Kaarshika/datasets/kaarshika_rl_preference_dataset_1000.jsonl"

In [5]:
dataset = load_dataset('json', data_files = dataset_path)

In [6]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['prompt', 'chosen', 'rejected'],
        num_rows: 1000
    })
})


In [7]:
dataset = dataset.shuffle(seed = 42)

In [8]:
dataset['train'][0]

{'prompt': 'Crop: soybean\nStage: maturity\nSoil moisture: low\nRain probability: moderate\nTemperature: moderate\nHumidity: moderate\nWater availability: adequate\nForecast rainfall: heavy\n\nWhich action is more appropriate?',
 'chosen': 'DELAY_IRRIGATION',
 'rejected': 'IRRIGATE'}

### Train-Test Split

In [52]:
dataset_split = dataset['train'].train_test_split(test_size = 0.05, seed = 42)
train_dataset = dataset_split['train']
test_dataset = dataset_split['test']

dataset_split

DatasetDict({
    train: Dataset({
        features: ['prompt', 'chosen', 'rejected'],
        num_rows: 950
    })
    test: Dataset({
        features: ['prompt', 'chosen', 'rejected'],
        num_rows: 50
    })
})

## Model & Tokenizer setup

In [53]:
model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels = 1)
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### Create Prompt-Chosen & Prompt-Rejected

In [54]:
def add_combined_columns(example):
    example["prompt_chosen"] = (
        example["prompt"] + "\nAction: " + example["chosen"]
    )

    example["prompt_rejected"] = (
        example["prompt"] + "\nAction: " + example["rejected"]
    )

    return example

In [55]:
train_dataset = train_dataset.map(add_combined_columns)
test_dataset = test_dataset.map(add_combined_columns)

Map:   0%|          | 0/950 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

## Tokenization

In [59]:
max_length = 512

In [60]:
def preprocess_function(examples):
    tokenized_chosen = tokenizer(examples['prompt_chosen'], truncation = True, max_length = max_length, padding = 'max_length')
    tokenized_rejected = tokenizer(examples['prompt_rejected'], truncation = True, max_length = max_length, padding = 'max_length')

    return {
        "input_ids_chosen": tokenized_chosen['input_ids'],
        "attention_mask_chosen": tokenized_chosen['attention_mask'],
        "input_ids_rejected": tokenized_rejected['input_ids'],
        "attention_mask_rejected": tokenized_rejected['attention_mask']
    }

In [61]:
train_dataset = train_dataset.map(
    preprocess_function,
    batched=True
)

test_dataset = test_dataset.map(
    preprocess_function,
    batched=True
)

Map:   0%|          | 0/950 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

## LoRA Configuration

In [62]:
peft_config = LoraConfig(
    task_type = TaskType.SEQ_CLS,
    inference_mode = False,
    r = 8,
    lora_alpha = 32,
    lora_dropout = 0.1,
    target_modules = ['q_lin', 'v_lin'],
)

## Reward Config

In [63]:
from trl import RewardConfig

In [64]:
training_args = RewardConfig(
    output_dir="/content/drive/MyDrive/Kaarshika/models/reward_model",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    learning_rate=2e-5,
    logging_steps=10,
    num_train_epochs = 2,
    eval_strategy = 'steps',
    eval_steps = 50,
    run_name="kaarshika-distilbert-reward-v1"
)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


## RewardTrainer

In [65]:
from trl import RewardTrainer

In [66]:
trainer = RewardTrainer(
    model=model,
    args=training_args,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    peft_config=peft_config,
)

/usr/local/lib/python3.12/dist-packages/trl/trainer/reward_trainer.py:182: UserWarning: When using RewardDataCollatorWithPadding, you should set `max_length` in RewardConfig. It will be set to `512` by default, but you should do it yourself in the future.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/reward_trainer.py:199: UserWarning: When using RewardDataCollatorWithPadding, you should set `remove_unused_columns=False` in your RewardConfig we have set it for you, but you should do it yourself in the future.
  warnings.warn(


In [67]:
output_dir = '/content/drive/MyDrive/Kaarshika/models/reward_model'

In [68]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [69]:
trainer.train()

You're using a DistilBertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:2847: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
Could not estimate the number of tokens of the input, floating-point operations will not be computed


Step,Training Loss,Validation Loss,Accuracy
50,0.686700,0.687080,0.680000


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ chosen_text                                   ┃ rejected_text                                ┃ logits           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ [CLS] crop : rice stage : grain filling soil  │ [CLS] crop : rice stage : grain filling soil │ [0.4933, 0.5067] │
│ moisture : high rain probability : high       │ moisture : high rain probability : high      │                  │
│ temperature : moderate humidity : high water  │ temperature : moderate humidity : high water │                  │
│ availability : adequate recent rainfall :     │ availability : adequate recent rainfall :    │                  │
│ heavy drainage condition : poor which action  │ heavy drainage condition : poor which action │                  │
│ is more appropriate? action : improve _       │ is more appropriate? action : monitor _ crop │                  │
│ drainage [SEP]                                │ [SEP]                                        │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ [CLS] crop : tomato stage : ripening soil     │ [CLS] crop : tomato stage : ripening soil    │ [0.5227, 0.4773] │
│ moisture : moderate rain probability : low    │ moisture : moderate rain probability : low   │                  │
│ temperature : moderate humidity : moderate    │ temperature : moderate humidity : moderate   │                  │
│ water availability : adequate which action is │ water availability : adequate which action   │                  │
│ more appropriate? action : reduce _           │ is more appropriate? action : irrigate [SEP] │                  │
│ irrigation [SEP]                              │                                              │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ [CLS] crop : maize stage : tasseling soil     │ [CLS] crop : maize stage : tasseling soil    │ [0.5012, 0.4988] │
│ moisture : moderate rain probability : low    │ moisture : moderate rain probability : low   │                  │
│ temperature : high humidity : low water       │ temperature : high humidity : low water      │                  │
│ availability : limited irrigation             │ availability : limited irrigation            │                  │
│ availability : limited which action is more   │ availability : limited which action is more  │                  │
│ appropriate? action : conserve _ water [SEP]  │ appropriate? action : monitor _ crop [SEP]   │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ [CLS] crop : groundnut stage : seedling soil  │ [CLS] crop : groundnut stage : seedling soil │ [0.5007, 0.4993] │
│ moisture : moderate rain probability : high   │ moisture : moderate rain probability : high  │                  │
│ temperature : low humidity : high water       │ temperature : low humidity : high water      │                  │
│ availability : adequate weather condition :   │ availability : adequate weather condition :  │                  │
│ cloudy which action is more appropriate?      │ cloudy which action is more appropriate?     │                  │
│ action : delay _ irrigation [SEP]             │ action : wait _ for _ rain [SEP]             │                  │
└───────────────────────────────────────────────┴──────────────────────────────────────────────┴──────────────────┘

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:2847: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


TrainOutput(global_step=58, training_loss=0.6895164210220863, metrics={'train_runtime': 136.6924, 'train_samples_per_second': 13.9, 'train_steps_per_second': 0.424, 'total_flos': 0.0, 'train_loss': 0.6895164210220863, 'epoch': 1.949579831932773})

In [70]:
trainer.save_model(output_dir)

---

# <b>Inference

## Loading the saved PEFT model

In [71]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import PeftModel

In [72]:
base_model = AutoModelForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels = 1
)

model = PeftModel.from_pretrained(
    base_model,
    output_dir
)

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

model.eval()

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): DistilBertForSequenceClassification(
      (distilbert): DistilBertModel(
        (embeddings): Embeddings(
          (word_embeddings): Embedding(30522, 768, padding_idx=0)
          (position_embeddings): Embedding(512, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (transformer): Transformer(
          (layer): ModuleList(
            (0-5): 6 x TransformerBlock(
              (attention): MultiHeadSelfAttention(
                (dropout): Dropout(p=0.1, inplace=False)
                (q_lin): lora.Linear(
                  (base_layer): Linear(in_features=768, out_features=768, bias=True)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.1, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=768

### Using the model for scoring purposes.

In [73]:
prompt = """
Crop: cotton
Stage: flowering
Soil moisture: low
Rain probability: low
Temperature: high
Humidity: low
Water availability: adequate
Which action is more appropriate?
"""

response1 = "WAIT_FOR_RAIN"

response2 = "IRRIGATE"

responses = [response1, response2]

scores = []

for i, response in enumerate(responses, 1):

    text = f"{prompt}\nAction: {response}"

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        score = model(**inputs).logits.item()

    scores.append(score)

    print(f"Action {i}: {response}")
    print(f"Reward Score: {score:.4f}\n")

best = scores.index(max(scores))

print(f"Best Action: {responses[best]}")

Action 1: WAIT_FOR_RAIN
Reward Score: 0.0463

Action 2: IRRIGATE
Reward Score: 0.0044

Best Action: WAIT_FOR_RAIN
